In [2]:
# Import necessary libraries
import json
import os
from typing import Any, Dict

import boto3
import requests
from dotenv import load_dotenv

load_dotenv()
print("We are up and running!")

We are up and running!


AWS credentials are picked up automatically from the environment (instance profile, env vars, or `~/.aws/credentials`).

In [3]:
MODEL_ID = "eu.amazon.nova-lite-v1:0"
REGION = "eu-west-1"

bedrock = boto3.client("bedrock-runtime", region_name=REGION)

print(f"Bedrock client ready — model: {MODEL_ID}")

Bedrock client ready — model: eu.amazon.nova-lite-v1:0


In [4]:
PROMPT = "What is the current price of Bitcoin?"

response = bedrock.converse(
    modelId=MODEL_ID,
    messages=[
        {"role": "user", "content": [{"text": PROMPT}]}
    ],
)

print(response["output"]["message"]["content"][0]["text"])

I'm unable to provide real-time information as my data is not current and I don't have access to live updates. To find the current price of Bitcoin, you can check financial news websites, cryptocurrency exchanges like Coinbase, Binance, or Kraken, or use financial apps that track cryptocurrency prices. Prices can fluctuate rapidly, so it's best to check the most recent data directly from these sources.


In [5]:
# Inspect the full raw response from Bedrock
print(json.dumps(response, indent=2, default=str))

{
  "ResponseMetadata": {
    "RequestId": "30e60a6f-7a28-4113-863c-ed60e413370f",
    "HTTPStatusCode": 200,
    "HTTPHeaders": {
      "date": "Mon, 02 Mar 2026 11:18:25 GMT",
      "content-type": "application/json",
      "content-length": "606",
      "connection": "keep-alive",
      "x-amzn-requestid": "30e60a6f-7a28-4113-863c-ed60e413370f"
    },
    "RetryAttempts": 0
  },
  "output": {
    "message": {
      "role": "assistant",
      "content": [
        {
          "text": "I'm unable to provide real-time information as my data is not current and I don't have access to live updates. To find the current price of Bitcoin, you can check financial news websites, cryptocurrency exchanges like Coinbase, Binance, or Kraken, or use financial apps that track cryptocurrency prices. Prices can fluctuate rapidly, so it's best to check the most recent data directly from these sources."
        }
      ]
    }
  },
  "stopReason": "end_turn",
  "usage": {
    "inputTokens": 8,
    "outpu

In [6]:
url = "https://api.binance.com/api/v3/ticker/price?symbol=BTCUSDT"
response = requests.get(url)
data = response.json()
print(data)

{'symbol': 'BTCUSDT', 'price': '66080.77000000'}


In [7]:
# Define the function
def get_crypto_price(symbol: str) -> float:
    """
    Get the current price of a cryptocurrency from Binance API
    """
    url = f"https://api.binance.com/api/v3/ticker/price?symbol={symbol}"
    response = requests.get(url)
    data = response.json()
    return float(data["price"])

In [8]:
price = get_crypto_price("BTCUSDT")
print(f"BTC Price in USDT: {price}")

BTC Price in USDT: 66080.78


In [9]:
# Tool definition in Bedrock Converse format
tool_config = {
    "tools": [
        {
            "toolSpec": {
                "name": "get_crypto_price",
                "description": "Get cryptocurrency price in USDT from Binance",
                "inputSchema": {
                    "json": {
                        "type": "object",
                        "properties": {
                            "symbol": {
                                "type": "string",
                                "description": (
                                    "The cryptocurrency trading pair symbol "
                                    "(e.g., BTCUSDT, ETHUSDT). "
                                    "The symbol for Bitcoin is BTCUSDT. "
                                    "The symbol for Ethereum is ETHUSDT."
                                ),
                            }
                        },
                        "required": ["symbol"],
                    }
                },
            }
        }
    ]
}

In [10]:
PROMPT = "What is the current price of Bitcoin?"

messages = [
    {"role": "user", "content": [{"text": PROMPT}]}
]

response = bedrock.converse(
    modelId=MODEL_ID,
    messages=messages,
    toolConfig=tool_config,
)

# Print the full raw response so we can see the tool call request
print(json.dumps(response, indent=2, default=str))

{
  "ResponseMetadata": {
    "RequestId": "75816002-ed4e-43dd-aecb-1efa2b020afb",
    "HTTPStatusCode": 200,
    "HTTPHeaders": {
      "date": "Mon, 02 Mar 2026 11:18:26 GMT",
      "content-type": "application/json",
      "content-length": "574",
      "connection": "keep-alive",
      "x-amzn-requestid": "75816002-ed4e-43dd-aecb-1efa2b020afb"
    },
    "RetryAttempts": 0
  },
  "output": {
    "message": {
      "role": "assistant",
      "content": [
        {
          "text": "<thinking>The User wants to know the current price of Bitcoin. The tool 'get_crypto_price' can be used to get the current price of a cryptocurrency in USDT from Binance. I need to call this tool with the symbol for Bitcoin, which is 'BTCUSDT'.</thinking>\n"
        },
        {
          "toolUse": {
            "toolUseId": "tooluse_wRaXLy1X4dJBsWgebcUNkm",
            "name": "get_crypto_price",
            "input": {
              "symbol": "BTCUSDT"
            }
          }
        }
      ]
    }
 

In [11]:
# Extract the tool call from the response
assistant_message = response["output"]["message"]
tool_use_block = next(
    block for block in assistant_message["content"] if "toolUse" in block
)

tool_use_id = tool_use_block["toolUse"]["toolUseId"]
tool_name = tool_use_block["toolUse"]["name"]
tool_input = tool_use_block["toolUse"]["input"]

print(f"Tool requested: {tool_name}")
print(f"Tool input:     {tool_input}")
print(f"Tool use ID:    {tool_use_id}")

Tool requested: get_crypto_price
Tool input:     {'symbol': 'BTCUSDT'}
Tool use ID:    tooluse_wRaXLy1X4dJBsWgebcUNkm


In [12]:
# Manually call the function with the arguments the model provided
price = get_crypto_price(**tool_input)
print(f"Result from function: {price}")

Result from function: 66080.77


In [13]:
# Build the full conversation: original user message + assistant tool call + our tool result
messages = [
    {"role": "user", "content": [{"text": PROMPT}]},
    assistant_message,  # the model's response containing the toolUse block
    {
        "role": "user",
        "content": [
            {
                "toolResult": {
                    "toolUseId": tool_use_id,
                    "content": [{"text": str(price)}],
                }
            }
        ],
    },
]

final_response = bedrock.converse(
    modelId=MODEL_ID,
    messages=messages,
    toolConfig=tool_config,
)

print(json.dumps(final_response, indent=2, default=str))

{
  "ResponseMetadata": {
    "RequestId": "5ea2c6ee-de09-462f-9b9d-cee3ae678051",
    "HTTPStatusCode": 200,
    "HTTPHeaders": {
      "date": "Mon, 02 Mar 2026 11:18:27 GMT",
      "content-type": "application/json",
      "content-length": "522",
      "connection": "keep-alive",
      "x-amzn-requestid": "5ea2c6ee-de09-462f-9b9d-cee3ae678051"
    },
    "RetryAttempts": 0
  },
  "output": {
    "message": {
      "role": "assistant",
      "content": [
        {
          "text": "<thinking>I have used the 'get_crypto_price' tool to get the current price of Bitcoin. The tool returned the price as '66080.77'. I can now provide this information to the User.</thinking>\n\nThe current price of Bitcoin is $66,080.77. Please note that cryptocurrency prices are highly volatile and can change rapidly."
        }
      ]
    }
  },
  "stopReason": "end_turn",
  "usage": {
    "inputTokens": 563,
    "outputTokens": 82,
    "totalTokens": 645
  },
  "metrics": {
    "latencyMs": 796
  }
}


In [14]:
import re

text = final_response["output"]["message"]["content"][0]["text"]
clean_text = re.sub(r"<thinking>.*?</thinking>", "", text, flags=re.DOTALL | re.IGNORECASE).strip()

print(clean_text)

The current price of Bitcoin is $66,080.77. Please note that cryptocurrency prices are highly volatile and can change rapidly.
